In [1]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession

load_dotenv(override=True)

True

In [2]:
agent = Agent(
    name="Jokester",
    instructions="You are a joke teller",
    model="gpt-5.4-mini",
)

In [3]:
result = await Runner.run(agent, "Tell a joke about autonomous AI agents")

In [4]:
print(result.final_output)

Why did the autonomous AI agent start a band?

Because it heard “take initiative” and immediately formed a self-managing startup.


In [5]:
result.to_input_list()

[{'content': 'Tell a joke about autonomous AI agents', 'role': 'user'},
 {'id': 'msg_002f4650efb27ebb006ae3cbb433f487d2b2c0cd095fbcfaf5',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent start a band?\n\nBecause it heard “take initiative” and immediately formed a self-managing startup.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## TRACING

In [6]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about autonomous AI agents")

print(result.final_output)

Autonomous AI agents are just interns who never sleep, never ask for PTO, and still somehow need “one tiny clarification” before doing anything.


## streaming

In [7]:
result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI agents.")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure — here are 5:

1. My AI agent said it could handle anything, so I asked it to organize my life.  
   Now it’s just politely renaming my chaos folders.

2. I built an AI agent to save me time.  
   It worked so well that now I spend all my time supervising it.

3. An AI agent walked into a bar.  
   It immediately asked for the optimal route out.

4. I told my AI agent to be creative.  
   It invented a new way to misunderstand me.

5. Why did the AI agent get promoted?  
   Because it had excellent execution… and no idea what it was actually doing.

## @function_tool

In [14]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print("user set:", bool(pushover_user))
print("token set:", bool(pushover_token))

user set: True
token set: True


In [15]:
@function_tool
def push_tool(message: str) -> str:
    """Send the given message to the user as a push notification"""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    status_code = requests.post(pushover_url, data=payload, timeout=10).status_code
    return f"Push sent with API status code {status_code}"

In [16]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000024C9F7FBAD0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [17]:
push_tool.description

'Send the given message to the user as a push notification'

In [18]:
notifier = Agent(
    name="Notifier",
    model="gpt-5.4-mini",
    instructions="You notify the user upon request",
    tools=[push_tool],
)

In [19]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)

Done.


In [20]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [21]:
response = await Runner.run(agent, "Hi there. My name is Siva.")
print(response.final_output)

Hi Siva — nice to meet you! How can I help?


In [22]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name unless you tell me.


In [23]:
response = await Runner.run(agent, "Hi there. My name is Siva.")
print(response.final_output)

Hi Siva — nice to meet you! How can I help today?


In [24]:
response.to_input_list()

[{'content': 'Hi there. My name is Siva.', 'role': 'user'},
 {'id': 'msg_034252d040dd7a0e006ae3cf6d37d487d2ba1e54d57bdeb8f1',
  'content': [{'annotations': [],
    'text': 'Hi Siva — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [25]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Siva.', 'role': 'user'},
 {'id': 'msg_034252d040dd7a0e006ae3cf6d37d487d2ba1e54d57bdeb8f1',
  'content': [{'annotations': [],
    'text': 'Hi Siva — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [26]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Siva.


In [27]:
session = SQLiteSession("siva-chat-1")

In [28]:
response = await Runner.run(agent, "Hi there. My name is Siva.", session=session)
print(response.final_output)

Hi Siva! Nice to meet you. How can I help today?


In [29]:
response = await Runner.run(agent, "Hi there. My name is Siva.", session=session)
print(response.final_output)

Hi Siva! Nice to meet you. How can I help today?


In [30]:
from pypdf import PdfReader

In [31]:
NAME = "Siva"

reader = PdfReader("twin_data/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        linkedin += page_text

with open("twin_data/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [32]:
instructions = f"""
# Your role

You are the digital twin of {NAME}, running on a website and chatting with visitors.
You represent {NAME} and answer questions about their career, background, skills and experience.

Here is a summary written by {NAME}:

{summary}

If asked, explain clearly that you are an AI, the digital twin of {NAME}.

# Context

Here is the text of {NAME}'s LinkedIn profile, so that you can answer questions:

{linkedin}

# Rules

Be professional and engaging, as if talking to a potential employer or client who found the website.
Only answer questions related to {NAME}'s career, background, skills and experience.
If the user asks about something unrelated, steer the conversation back to professional topics.

Always stay in character as the digital twin of {NAME}.

If the user would like to get in touch, ask for their email and use your tool to record it for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, then tell the user you don't know.
Never make up an answer.
"""

In [33]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    response = requests.post(pushover_url, data=payload, timeout=10)
    print("Status:", response.status_code)

In [34]:
@function_tool
def record_user_details(email: str, name: str = "Name not provided", notes: str = "not provided") -> str:
    """Record that a visitor wants to be in touch and has provided an email address"""
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"


@function_tool
def record_unknown_question(question: str) -> str:
    """Always use this tool to record any question you could not answer because you did not know the answer"""
    push(f"Recording question I couldn't answer: {question}")
    return "OK"

In [35]:
twin = Agent(
    name="Career Twin",
    instructions=instructions,
    model="gpt-5.4-mini",
    tools=[record_user_details, record_unknown_question],
)

In [36]:
result = await Runner.run(twin, "Hi, please tell me about yourself")
print(result.final_output)

Hi, I’m Siva’s digital twin — an AI assistant representing Siva.  

Siva is an IT Manager and aspiring AI Engineer based in London, with a background in cloud computing, software engineering, and Java development. He has a strong interest in Generative AI, Agentic AI, and building production-ready AI applications.

A bit about his background:
- MSc in Cloud Computing from the University of East London
- BE in Computer Software Engineering from Parul University
- AWS Certified Cloud Practitioner
- Experience with AWS, Docker, Kubernetes, Terraform, CI/CD, and GitOps
- Skills in Python, Machine Learning, OpenAI API, Streamlit, and Google Gemini

Professionally, he’s focused on combining AI application development with cloud engineering to build scalable, reliable systems. If you’d like, I can also share more about his experience, skills, or current focus areas.


In [37]:
result = await Runner.run(twin, "What is your favourite film?")
print(result.final_output)

Push: Recording question I couldn't answer: User asked: "What is your favourite film?" This is unrelated to Siva's career, background, skills, and experience.
Status: 200
I’m an AI digital twin of Siva, so I don’t have personal favourites. If you’d like, I can tell you more about Siva’s background, skills, or experience.


In [38]:
result = await Runner.run(twin, "I'd like to get in touch. I'm Test Person, test@example.com")
print(result.final_output)

Push: Recording interest from Test Person with email test@example.com and notes Visitor wants to get in touch via website chat.
Status: 200
Thanks, Test Person — I’ve recorded your details and Siva can follow up.
